In [10]:
# =========================
# node2vec demo in a supply-chain (firm-to-firm) network domain
# =========================
# What this code does
# 1) Builds a homogeneous directed supply graph where nodes are firms and edges mean:
#       "supplier -> buyer"  (ships-to / supplies-to)
#    This is naturally homogeneous: only one node type (Firm) and one edge meaning (Supply link).
# 2) Trains node2vec to learn firm embeddings from random-walk co-occurrence in the supply network.
# 3) Demonstrates "proposal generation" patterns (soft reasoning / candidate generation):
#    A) Similar suppliers to a given supplier (substitutes / peer suppliers)
#    B) Similar buyers to a given buyer (peer buyers / comparable demand profiles)
#    C) Candidate alternative suppliers for a specific buyer:
#         - compute an "ideal supplier profile" as centroid of the buyer's current suppliers
#         - rank other suppliers by similarity to that profile
#    D) Risk propagation candidate set:
#         - given a disrupted supplier, propose likely impacted firms by embedding proximity

In [11]:
# Install:
#   pip install networkx node2vec gensim numpy

import networkx as nx
import numpy as np
from node2vec import Node2Vec

In [12]:
# -------------------------
# A. Build a supply-chain network (Firm -> Firm)
# -------------------------
# Nodes are firms. A directed edge A -> B means "A supplies B".
# We keep it directed because direction carries meaning in supply chains.
# (node2vec can run on directed graphs; NetworkX DiGraph is fine.)
# NetworkX is a python package for creating and manipulating networks (aka graphs aka knowledge graphs) 

firms = [
    # upstream suppliers
    "S:AlphaMetals", "S:BetaPlastics", "S:GammaChips", "S:DeltaSensors", "S:EpsilonChem",
    "S:OmegaLogistics",
    # midstream manufacturers
    "M:AuroraDevices", "M:BorealRobotics", "M:CascadeAuto", "M:DriftAero",
    # downstream assemblers / brands
    "B:NorthstarElectronics", "B:OrionMobility", "B:PolarisAviation",
    # some additional firms
    "S:KappaChips", "S:LambdaSensors", "M:HeliosIndustrial", "B:VegaConsumer",
]

edges = [
    # Core supply to manufacturers
    ("S:AlphaMetals", "M:AuroraDevices"),
    ("S:AlphaMetals", "M:CascadeAuto"),
    ("S:BetaPlastics", "M:AuroraDevices"),
    ("S:BetaPlastics", "M:BorealRobotics"),
    ("S:GammaChips", "M:AuroraDevices"),
    ("S:GammaChips", "M:BorealRobotics"),
    ("S:DeltaSensors", "M:BorealRobotics"),
    ("S:DeltaSensors", "M:DriftAero"),
    ("S:EpsilonChem", "M:CascadeAuto"),
    ("S:EpsilonChem", "M:DriftAero"),

    # Alternative suppliers with overlapping coverage
    ("S:KappaChips", "M:AuroraDevices"),
    ("S:KappaChips", "M:HeliosIndustrial"),
    ("S:LambdaSensors", "M:BorealRobotics"),
    ("S:LambdaSensors", "M:DriftAero"),

    # Logistics touches many 
    ("S:OmegaLogistics", "M:AuroraDevices"),
    ("S:OmegaLogistics", "M:BorealRobotics"),
    ("S:OmegaLogistics", "M:CascadeAuto"),
    ("S:OmegaLogistics", "M:DriftAero"),
    ("S:OmegaLogistics", "M:HeliosIndustrial"),

    # Manufacturers supply brands
    ("M:AuroraDevices", "B:NorthstarElectronics"),
    ("M:BorealRobotics", "B:NorthstarElectronics"),
    ("M:CascadeAuto", "B:OrionMobility"),
    ("M:DriftAero", "B:PolarisAviation"),
    ("M:HeliosIndustrial", "B:VegaConsumer"),

    # Some cross-supply relationships
    ("S:AlphaMetals", "M:HeliosIndustrial"),
    ("S:BetaPlastics", "M:HeliosIndustrial"),
    ("M:AuroraDevices", "B:VegaConsumer"),
    ("M:BorealRobotics", "B:OrionMobility"),
]

G = nx.DiGraph()
G.add_nodes_from(firms)
G.add_edges_from(edges)

print(f"Supply graph: nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

Supply graph: nodes=17 edges=28


In [13]:
# -------------------------
# B. Train node2vec embeddings on the supply network
# -------------------------
# Walks move along directed supply links, producing sequences like:
# Supplier -> Manufacturer -> Brand
# and also via shared suppliers/logistics connections.
# Skip-gram then learns that firms appearing in similar "supply contexts" get similar vectors.

node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=18,
    num_walks=120,
    p=1.0,
    q=1.0,
    workers=2,
    seed=42,
)

w2v = node2vec.fit(
    window=8,
    min_count=1,
    batch_words=256,
)

Computing transition probabilities:   0%|          | 0/17 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 60/60 [00:00<00:00, 22788.94it/s]


In [14]:
# -------------------------
# C. Helper functions: cosine similarity + filtered nearest neighbors
# -------------------------
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(np.dot(a, b))

def firm_type(f: str) -> str:
    return f.split(":", 1)[0]  # S / M / B

def nearest_firms(seed: str, topn: int = 12, restrict_prefix: str | None = None):
    """
    Return nearest neighbors for a firm.
    If restrict_prefix is provided (e.g., "S"), only return firms of that prefix.
    """
    sims = w2v.wv.most_similar(seed, topn=topn * 3)
    out = []
    for n, s in sims:
        if restrict_prefix is None or firm_type(n) == restrict_prefix:
            out.append((n, float(s)))
        if len(out) >= topn:
            break
    return out

def centroid(nodes):
    """Vector centroid of a list of node IDs."""
    vecs = [w2v.wv[n] for n in nodes if n in w2v.wv]
    return np.mean(np.vstack(vecs), axis=0)

def rank_by_centroid(c, candidates):
    """Rank candidate firms by similarity to a centroid vector."""
    scored = [(x, cosine(c, w2v.wv[x])) for x in candidates if x in w2v.wv]
    return sorted(scored, key=lambda t: t[1], reverse=True)

In [16]:
# D. Demos

# -------------------------
# 1: peer suppliers / substitute suppliers (proposal generation)
# -------------------------
seed_supplier = "S:GammaChips"
print("\n=== peer suppliers to a supplier (substitutes / comparable suppliers) ===")
for f, s in nearest_firms(seed_supplier, topn=8, restrict_prefix="S"):
    print(f"{seed_supplier:15s} ~ {f:15s}  sim={s:.3f}")


# -------------------------
# 2: peer buyers / comparable brands
# -------------------------
seed_brand = "B:NorthstarElectronics"
print("\n=== peer buyers to a buyer (brands with similar upstream contexts) ===")
for f, s in nearest_firms(seed_brand, topn=8, restrict_prefix="B"):
    print(f"{seed_brand:22s} ~ {f:22s}  sim={s:.3f}")


# -------------------------
# 3: propose alternative suppliers for a given manufacturer or buyer
# -------------------------
# Idea:
# - If a buyer/manufacturer has a set of current suppliers, those suppliers define a "supply profile".
# - We form a centroid of current suppliers' embeddings.
# - Then we rank other suppliers by similarity to that centroid.
# This is a common "candidate generation" move: propose plausible substitutes.

target_manufacturer = "M:AuroraDevices"
current_suppliers = list(G.predecessors(target_manufacturer))  # incoming edges are suppliers
supplier_candidates = [n for n in firms if firm_type(n) == "S" and n not in current_suppliers]

print("\n===  propose alternative suppliers for a manufacturer ===")
print("Target manufacturer:", target_manufacturer)
print("Current suppliers:", current_suppliers)

profile = centroid(current_suppliers)
ranked_alternatives = rank_by_centroid(profile, supplier_candidates)[:10]

print("\nTop proposed alternative suppliers (not currently supplying the target):")
for f, s in ranked_alternatives:
    print(f"candidate={f:15s}  sim_to_supplier_profile={s:.3f}")


# -------------------------
# 4: propose likely impacted firms from a disrupted supplier (soft blast-radius)
# -------------------------
# Idea:
# - Given a disrupted supplier, firms close in embedding space often share neighborhood context:
#   similar customers, same tier, shared downstream paths.
# - We propose impacted firms as nearest neighbors (across all firm types), then you can
#   hand these candidates to whatever deeper analysis you like (simulation, inventory checks, etc.)

disrupted_supplier = "S:DeltaSensors"
print("\n=== likely impacted firms from a disrupted supplier (candidate set) ===")
neighbors = w2v.wv.most_similar(disrupted_supplier, topn=15)

for f, s in neighbors:
    print(f"{disrupted_supplier:15s} ~ {f:22s}  type={firm_type(f)}  sim={float(s):.3f}")


# -------------------------
# show that embeddings capture "directional roles" (supplier vs buyer vs manufacturer)
# -------------------------
# A quick sanity check: nearest neighbors of a supplier often include its customers (manufacturers),
# and suppliers that share customers; nearest neighbors of a brand include upstream manufacturers.

print("\n=== Sanity check: neighborhood mix for one firm ===")
probe = "M:BorealRobotics"
for f, s in w2v.wv.most_similar(probe, topn=12):
    print(f"{probe:16s} ~ {f:22s}  type={firm_type(f)}  sim={float(s):.3f}")


=== peer suppliers to a supplier (substitutes / comparable suppliers) ===
S:GammaChips    ~ S:AlphaMetals    sim=0.080
S:GammaChips    ~ S:EpsilonChem    sim=0.072
S:GammaChips    ~ S:DeltaSensors   sim=0.056
S:GammaChips    ~ S:BetaPlastics   sim=0.054
S:GammaChips    ~ S:OmegaLogistics  sim=0.041
S:GammaChips    ~ S:LambdaSensors  sim=0.011
S:GammaChips    ~ S:KappaChips     sim=-0.015

=== peer buyers to a buyer (brands with similar upstream contexts) ===
B:NorthstarElectronics ~ B:OrionMobility         sim=0.361
B:NorthstarElectronics ~ B:VegaConsumer          sim=0.144
B:NorthstarElectronics ~ B:PolarisAviation       sim=0.141

===  propose alternative suppliers for a manufacturer ===
Target manufacturer: M:AuroraDevices
Current suppliers: ['S:AlphaMetals', 'S:BetaPlastics', 'S:GammaChips', 'S:KappaChips', 'S:OmegaLogistics']

Top proposed alternative suppliers (not currently supplying the target):
candidate=S:LambdaSensors  sim_to_supplier_profile=0.076
candidate=S:EpsilonChem  